In [1]:
from pathlib import Path
from difflib import SequenceMatcher
import json
import re
import requests
import numpy as np
import pandas as pd

PROCESSED_DIR = Path("data/processed")
INTERIM_DIR = Path("data/interim")
TABLES_DIR = Path("outputs/tables")
REFERENCE_DATE = pd.Timestamp("2026-07-16")

for folder in [PROCESSED_DIR, INTERIM_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

priced = pd.read_csv(PROCESSED_DIR / "pokemon_priced_cards.csv", dtype={"Card Number": str}, low_memory=False)
unpriced = pd.read_csv(PROCESSED_DIR / "pokemon_unpriced_cards.csv", dtype={"Card Number": str}, low_memory=False)

priced["priced_row"] = 1
unpriced["priced_row"] = 0
all_cards = pd.concat([priced, unpriced], ignore_index=True, sort=False)

In [2]:
def clean_text(value):
    if pd.isna(value):
        return ""
    value = str(value).lower().replace("é", "e")
    return re.sub(r"[^a-z0-9]+", " ", value).strip()

def combined_text(df, columns):
    return df[columns].fillna("").astype(str).agg(" ".join, axis=1).map(clean_text)

all_cards["Card Name_clean"] = all_cards["Card Name"].map(clean_text)

all_cards["HP"] = pd.to_numeric(all_cards["HP"], errors="coerce")
all_cards["HP_missing"] = all_cards["HP"].isna().astype(int)
all_cards["HP_filled"] = all_cards["HP"].fillna(0)

card_text = combined_text(all_cards, ["Card Name", "Card Type", "Rarity"])
all_cards["Card Category"] = np.select(
    [card_text.str.contains(r"\btrainer\b"), card_text.str.contains(r"\benergy\b"), all_cards["HP"].notna()],
    ["Trainer", "Energy", "Pokemon"],
    default="Other"
)

all_cards["Card Category V2"] = np.select(
    [
        card_text.str.contains("basic energy"), card_text.str.contains("special energy"),
        card_text.str.contains("stadium"), card_text.str.contains("supporter"),
        card_text.str.contains(r"\bitem\b"), card_text.str.contains(r"\btrainer\b"),
        all_cards["Card Category"].eq("Pokemon"), card_text.str.contains(r"\benergy\b")
    ],
    ["Basic Energy", "Special Energy", "Stadium Trainer", "Supporter Trainer",
     "Item Trainer", "Trainer", "Pokemon", "Energy"],
    default=all_cards["Card Category"]
)

In [3]:
def rarity_group(value):
    text = clean_text(value)
    if not text:
        return "Unknown"
    if any(x in text for x in ["secret", "rainbow", "hyper", "gold", "special illustration", "illustration rare"]):
        return "Secret/Special Rare"
    if any(x in text for x in ["ultra", "full art", "vmax", "vstar", " gx", " ex"]):
        return "Ultra Rare"
    if any(x in text for x in ["holo", "foil"]):
        return "Holo Rare"
    if "promo" in text:
        return "Promo"
    if "uncommon" in text:
        return "Uncommon"
    if "common" in text:
        return "Common"
    if "rare" in text:
        return "Rare"
    return "Other"

rarity_rank = {
    "Unknown": 0, "Common": 1, "Uncommon": 2, "Other": 2,
    "Rare": 3, "Holo Rare": 4, "Promo": 4,
    "Ultra Rare": 5, "Secret/Special Rare": 6
}

all_cards["rarity_group"] = all_cards["Rarity"].map(rarity_group)
all_cards["rarity_rank"] = all_cards["rarity_group"].map(rarity_rank).astype(int)

number = all_cards["Card Number"].fillna("").astype(str).str.strip().str.upper()
all_cards["card_number_missing"] = number.eq("").astype(int)
all_cards["card_number_numeric"] = number.str.extract(r"(\d+)", expand=False).astype(float)
all_cards["card_number_printed_total"] = number.str.extract(r"/[A-Z]*(\d+)", expand=False).astype(float)
all_cards["card_number_has_letters"] = number.str.contains(r"[A-Z]").astype(int)
all_cards["card_number_prefix"] = number.str.extract(r"^([A-Z]+)", expand=False).fillna("None")
all_cards["card_number_numeric_filled"] = all_cards["card_number_numeric"].fillna(0)

ratio = (all_cards["card_number_numeric"] / all_cards["card_number_printed_total"]).replace([np.inf, -np.inf], np.nan)
all_cards["card_position_ratio_intrinsic"] = ratio
all_cards["card_position_ratio_intrinsic_missing"] = ratio.isna().astype(int)
all_cards["is_secret_rare_intrinsic"] = (
    all_cards["card_number_numeric"].notna()
    & all_cards["card_number_printed_total"].notna()
    & (all_cards["card_number_numeric"] > all_cards["card_number_printed_total"])
).astype(int)

In [4]:
flag_text = combined_text(all_cards, ["Card Name", "Rarity", "Card Type", "Card Number", "Collection", "Set Name"])

flag_words = {
    "is_holo": ["holo", "holographic", "foil"],
    "is_reverse_holo": ["reverse holo", "reverse foil", "rev holo"],
    "is_non_holo": ["non holo", "nonholo", "non foil"],
    "is_alternate_art": ["alternate art", "alt art", "special art", "illustration rare", "special illustration"],
    "is_1st_edition": ["1st edition", "first edition"],
    "is_unlimited": ["unlimited"],
    "is_error_card": ["error", "misprint", "miscut", "mis cut"],
    "is_stamp_card": ["stamp", "stamped", "staff stamp", "winner stamp"],
    "is_prerelease": ["prerelease", "pre release"],
    "is_staff": ["staff"],
    "is_champion": ["champion", "championship", "champions festival"],
    "is_trainer_deck": ["trainer deck", "theme deck"],
    "is_world_championship": ["world championship", "world championships", "wcs"],
    "is_oversize": ["oversize", "oversized", "jumbo"],
    "is_prototype": ["prototype", "sample", "test print"],
    "is_promo_set": ["promo", "black star", "league", "players club"]
}

for feature, words in flag_words.items():
    all_cards[feature] = flag_text.map(lambda text: int(any(word in text for word in words)))

all_cards["is_holo"] = ((all_cards["is_holo"] == 1) | (all_cards["is_reverse_holo"] == 1)).astype(int)

names = all_cards["Card Name_clean"]
popular = ["charizard", "pikachu", "mew", "mewtwo", "rayquaza", "lugia", "ho oh", "gengar", "umbreon", "espeon", "eevee", "dragonite", "gyarados", "blastoise", "venusaur", "snorlax", "greninja", "lucario", "arceus", "giratina", "dialga", "palkia", "sylveon", "leafeon", "glaceon", "vaporeon", "jolteon", "flareon"]
eeveelutions = ["eevee", "vaporeon", "jolteon", "flareon", "espeon", "umbreon", "leafeon", "glaceon", "sylveon"]
legendary = ["mew", "mewtwo", "lugia", "ho oh", "celebi", "rayquaza", "jirachi", "deoxys", "dialga", "palkia", "giratina", "darkrai", "shaymin", "arceus", "reshiram", "zekrom", "kyurem", "keldeo", "genesect", "xerneas", "yveltal", "zygarde", "diancie", "hoopa", "volcanion", "solgaleo", "lunala", "necrozma", "marshadow", "zeraora", "zacian", "zamazenta", "eternatus", "calyrex", "miraidon", "koraidon"]

all_cards["is_popular_pokemon"] = names.map(lambda text: int(any(name in text for name in popular)))
all_cards["is_charizard"] = names.str.contains("charizard").astype(int)
all_cards["is_pikachu"] = names.str.contains("pikachu").astype(int)
all_cards["is_eeveelution"] = names.map(lambda text: int(any(name in text for name in eeveelutions)))
all_cards["is_legendary_or_mythical"] = names.map(lambda text: int(any(name in text for name in legendary)))

In [5]:
response = requests.get("https://api.pokemontcg.io/v2/sets", params={"pageSize": 250}, timeout=30)
response.raise_for_status()
sets = pd.json_normalize(response.json()["data"])

sets = sets.rename(columns={
    "id": "api_set_id", "name": "api_set_name", "series": "api_series",
    "releaseDate": "api_release_date", "printedTotal": "api_printed_total",
    "total": "api_total_cards"
})
sets["set_key"] = sets["api_set_name"].map(clean_text)
sets = sets.drop_duplicates("set_key")
sets.to_csv(INTERIM_DIR / "pokemon_tcg_sets_cache.csv", index=False, encoding="utf-8-sig")

set_lookup = sets.set_index("set_key").to_dict("index")
set_keys = list(set_lookup)

aliases = {
    "base": "base set",
    "base set 1": "base set",
    "jungle unlimited": "jungle",
    "fossil unlimited": "fossil",
    "team rocket unlimited": "team rocket",
    "bw promos": "bw black star promos",
    "xy promos": "xy black star promos",
    "sm promos": "sm black star promos",
    "swsh promos": "swsh black star promos",
    "sv promos": "scarlet violet black star promos",
    "scarlet violet promos": "scarlet violet black star promos"
}

def match_set(name):
    original_key = clean_text(name)
    key = aliases.get(original_key, original_key)

    if key in set_lookup:
        result = set_lookup[key].copy()
        result["match_type"] = "alias" if key != original_key else "exact"
        result["match_score"] = 1.0
        return result

    scores = [(candidate, SequenceMatcher(None, key, candidate).ratio()) for candidate in set_keys]
    best_key, best_score = max(scores, key=lambda x: x[1])

    if best_score >= .86:
        result = set_lookup[best_key].copy()
        result["match_type"] = "fuzzy"
        result["match_score"] = best_score
        return result

    return {"match_type": "unmatched", "match_score": best_score}

unique_sets = pd.DataFrame({"Set Name": all_cards["Set Name"].dropna().unique()})
matched = unique_sets["Set Name"].map(match_set).apply(pd.Series)
set_matches = pd.concat([unique_sets, matched], axis=1)

all_cards = all_cards.merge(set_matches, on="Set Name", how="left")

quality_rows = []
for name, data in [
    ("All cards", all_cards),
    ("Priced cards", all_cards[all_cards["priced_row"] == 1]),
    ("Unpriced cards", all_cards[all_cards["priced_row"] == 0]),
    ("Unique set names", set_matches)
]:
    quality_rows.append({
        "group": name, "rows": len(data),
        "match_rate_pct": data["api_set_id"].notna().mean() * 100
    })
quality = pd.DataFrame(quality_rows)

unmatched_sets = set_matches[set_matches["api_set_id"].isna()].sort_values("Set Name")
fuzzy_matches = set_matches[set_matches["match_type"].eq("fuzzy")].sort_values("match_score")

display(quality)
display(unmatched_sets.head(30))
display(fuzzy_matches.head(30))

display(set_matches["match_type"].value_counts().rename_axis("match_type").reset_index(name="set_count"))

set_matches.to_csv(TABLES_DIR / "set_api_matches.csv", index=False, encoding="utf-8-sig")
quality.to_csv(TABLES_DIR / "set_api_match_quality.csv", index=False, encoding="utf-8-sig")
unmatched_sets.to_csv(TABLES_DIR / "set_api_unmatched.csv", index=False, encoding="utf-8-sig")
fuzzy_matches.to_csv(TABLES_DIR / "set_api_fuzzy_matches.csv", index=False, encoding="utf-8-sig")

,group,rows,match_rate_pct
0,All cards,40455,92.646150
1,Priced cards,35732,93.009068
2,Unpriced cards,4723,89.900487
3,Unique set names,181,81.767956


,Set Name,api_set_id,api_set_name,api_series,api_printed_total,api_total_cards,ptcgoCode,api_release_date,updatedAt,legalities.unlimited,images.symbol,images.logo,legalities.expanded,legalities.standard,match_type,match_score
89,BW Miscellaneous Cards,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched,0.540541
76,Black & White Trainer Kit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched,0.647059
17,Box Topper,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched,0.461538
64,DP Miscellaneous Cards,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched,0.486486
48,Diamond & Pearl Trainer Kit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched,0.684211
30,EX Deoxys,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched,0.800000
22,EX Dragon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched,0.800000
31,EX Emerald,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched,0.823529
21,EX Sandstorm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched,0.857143
25,EX Trainer Kit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unmatched,0.800000


,Set Name,api_set_id,api_set_name,api_series,api_printed_total,api_total_cards,ptcgoCode,api_release_date,updatedAt,legalities.unlimited,images.symbol,images.logo,legalities.expanded,legalities.standard,match_type,match_score
69,Triumphant,hgss4,HS—Triumphant,HeartGold & SoulSilver,102.0,103.0,TM,2010/11/03,2018/03/04 10:35:00,Legal,https://images.pokemontcg.io/hgss4/symbol.png,https://images.pokemontcg.io/hgss4/logo.png,NaN,NaN,fuzzy,0.869565
0,Base Set,base4,Base Set 2,Base,130.0,130.0,B2,2000/02/24,2022/10/10 15:12:00,Legal,https://images.pokemontcg.io/base4/symbol.png,https://images.pokemontcg.io/base4/logo.png,NaN,NaN,fuzzy,0.888889
35,EX Legend Maker,ex12,Legend Maker,EX,92.0,93.0,LM,2006/02/01,2018/03/04 10:35:00,Legal,https://images.pokemontcg.io/ex12/symbol.png,https://images.pokemontcg.io/ex12/logo.png,NaN,NaN,fuzzy,0.888889
74,McDonald's Collection,mcd11,McDonald's Collection 2011,Other,12.0,12.0,NaN,2011/06/17,2022/10/10 15:12:00,Legal,https://images.pokemontcg.io/mcd11/symbol.png,https://images.pokemontcg.io/mcd11/logo.png,Legal,NaN,fuzzy,0.893617
42,EX Power Keepers,ex16,Power Keepers,EX,108.0,108.0,PK,2007/02/02,2018/03/04 10:35:00,Legal,https://images.pokemontcg.io/ex16/symbol.png,https://images.pokemontcg.io/ex16/logo.png,NaN,NaN,fuzzy,0.896552
20,EX Ruby & Sapphire,ex1,Ruby & Sapphire,EX,109.0,109.0,RS,2003/07/01,2018/03/04 10:35:00,Legal,https://images.pokemontcg.io/ex1/symbol.png,https://images.pokemontcg.io/ex1/logo.png,NaN,NaN,fuzzy,0.896552
32,EX Unseen Forces,ex10,Unseen Forces,EX,115.0,145.0,UF,2005/08/01,2020/08/14 09:35:00,Legal,https://images.pokemontcg.io/ex10/symbol.png,https://images.pokemontcg.io/ex10/logo.png,NaN,NaN,fuzzy,0.896552
34,EX Delta Species,ex11,Delta Species,EX,113.0,114.0,DS,2005/10/31,2020/05/01 16:06:00,Legal,https://images.pokemontcg.io/ex11/symbol.png,https://images.pokemontcg.io/ex11/logo.png,NaN,NaN,fuzzy,0.896552
38,EX Holon Phantoms,ex13,Holon Phantoms,EX,110.0,111.0,HP,2006/05/01,2018/03/04 10:35:00,Legal,https://images.pokemontcg.io/ex13/symbol.png,https://images.pokemontcg.io/ex13/logo.png,NaN,NaN,fuzzy,0.903226
24,EX Hidden Legends,ex5,Hidden Legends,EX,101.0,102.0,HL,2004/06/01,2019/01/28 16:44:00,Legal,https://images.pokemontcg.io/ex5/symbol.png,https://images.pokemontcg.io/ex5/logo.png,NaN,NaN,fuzzy,0.903226


,match_type,set_count
0,exact,130
1,unmatched,33
2,fuzzy,18


In [6]:
for column in ["api_printed_total", "api_total_cards"]:
    all_cards[column] = pd.to_numeric(all_cards[column], errors="coerce")
    all_cards[column + "_missing"] = all_cards[column].isna().astype(int)
    all_cards[column + "_filled"] = all_cards[column].fillna(0)

release_date = pd.to_datetime(all_cards["api_release_date"], errors="coerce")
all_cards["set_age_missing"] = release_date.isna().astype(int)
all_cards["set_age_filled"] = ((REFERENCE_DATE - release_date).dt.days / 365.25).clip(lower=0).fillna(0)

def set_era(date, series):
    if pd.isna(date):
        return clean_text(series).title() or "Unknown"
    year = date.year
    if year <= 2002: return "Wizards/Classic"
    if year <= 2007: return "EX Era"
    if year <= 2010: return "Diamond & Pearl/Platinum"
    if year <= 2013: return "Black & White"
    if year <= 2016: return "XY"
    if year <= 2019: return "Sun & Moon"
    if year <= 2022: return "Sword & Shield"
    return "Scarlet & Violet"

all_cards["set_era"] = [set_era(date, series) for date, series in zip(release_date, all_cards["api_series"])]

all_cards["card_position_ratio_api_printed"] = (
    all_cards["card_number_numeric"] / all_cards["api_printed_total"]
).replace([np.inf, -np.inf], np.nan)

all_cards["card_position_ratio_api_total"] = (
    all_cards["card_number_numeric"] / all_cards["api_total_cards"]
).replace([np.inf, -np.inf], np.nan)

all_cards["is_above_api_printed_total"] = (
    all_cards["card_number_numeric"].notna()
    & all_cards["api_printed_total"].notna()
    & (all_cards["card_number_numeric"] > all_cards["api_printed_total"])
).astype(int)

all_cards["is_above_api_total_cards"] = (
    all_cards["card_number_numeric"].notna()
    & all_cards["api_total_cards"].notna()
    & (all_cards["card_number_numeric"] > all_cards["api_total_cards"])
).astype(int)

interactions = {
    "rarity_rank_x_set_age": "rarity_rank",
    "holo_x_set_age": "is_holo",
    "secret_rare_x_set_age": "is_secret_rare_intrinsic",
    "popular_pokemon_x_set_age": "is_popular_pokemon",
    "stamp_card_x_set_age": "is_stamp_card",
    "first_edition_x_set_age": "is_1st_edition"
}
for feature, column in interactions.items():
    all_cards[feature] = all_cards[column] * all_cards["set_age_filled"]

In [7]:
intrinsic_features = [
    "Card Type", "Card Category", "Card Category V2", "HP_filled", "HP_missing",
    "Rarity", "rarity_group", "rarity_rank",
    "card_number_missing", "card_number_numeric_filled",
    "card_number_printed_total", "card_number_has_letters", "card_number_prefix",
    "card_position_ratio_intrinsic", "card_position_ratio_intrinsic_missing",
    "is_secret_rare_intrinsic",
    "is_holo", "is_reverse_holo", "is_non_holo", "is_alternate_art",
    "is_1st_edition", "is_unlimited", "is_error_card", "is_stamp_card",
    "is_prerelease", "is_staff", "is_champion", "is_trainer_deck",
    "is_world_championship", "is_oversize", "is_prototype", "is_promo_set",
    "is_popular_pokemon", "is_charizard", "is_pikachu", "is_eeveelution",
    "is_legendary_or_mythical"
]

extrinsic_features = [
    "api_series", "set_era", "set_age_filled", "set_age_missing",
    "api_printed_total_filled", "api_printed_total_missing",
    "api_total_cards_filled", "api_total_cards_missing",
    "card_position_ratio_api_printed", "card_position_ratio_api_total",
    "is_above_api_printed_total", "is_above_api_total_cards"
]

extrinsic_interactions = [
    "rarity_rank_x_set_age", "holo_x_set_age", "secret_rare_x_set_age",
    "popular_pokemon_x_set_age", "stamp_card_x_set_age", "first_edition_x_set_age"
]

feature_groups = {
    "intrinsic_features": intrinsic_features,
    "extrinsic_features": extrinsic_features,
    "extrinsic_interaction_features": extrinsic_interactions,
    "excluded_identity_features": [
        "Card Name", "Card Number", "Illustrator", "Set Name", "Collection"
    ]
}

feature_summary = pd.DataFrame([
    {"feature_group": group, "feature_count": len(features), "features": ", ".join(features)}
    for group, features in feature_groups.items()
])
display(feature_summary)

,feature_group,feature_count,features
0,intrinsic_features,37,"Card Type, Card Category, Card Category V2, HP..."
1,extrinsic_features,12,"api_series, set_era, set_age_filled, set_age_m..."
2,extrinsic_interaction_features,6,"rarity_rank_x_set_age, holo_x_set_age, secret_..."
3,excluded_identity_features,5,"Card Name, Card Number, Illustrator, Set Name,..."


In [8]:
priced_features = all_cards[all_cards["priced_row"] == 1].drop(columns="priced_row").copy()
unpriced_features = all_cards[all_cards["priced_row"] == 0].drop(columns=["priced_row", "Log_Value"], errors="ignore").copy()

priced_features.to_csv(PROCESSED_DIR / "pokemon_priced_features.csv", index=False, encoding="utf-8-sig")
unpriced_features.to_csv(PROCESSED_DIR / "pokemon_unpriced_features.csv", index=False, encoding="utf-8-sig")
all_cards.to_csv(INTERIM_DIR / "pokemon_all_features.csv", index=False, encoding="utf-8-sig")
feature_summary.to_csv(TABLES_DIR / "feature_groups_summary.csv", index=False, encoding="utf-8-sig")

with open(PROCESSED_DIR / "feature_groups.json", "w", encoding="utf-8") as file:
    json.dump(feature_groups, file, indent=2)

print("Priced features:", priced_features.shape)
print("Unpriced features:", unpriced_features.shape)

Priced features: (35732, 80)
Unpriced features: (4723, 79)
